In [97]:
import pandas as pd
import networkx as nx
from matplotlib import pyplot as plt
import numpy as np
import torch

In [98]:
edge_list = pd.read_csv(r"D:\commo\code\4_kumu_struct\edge_list.csv")
node1 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list1.csv")
node2 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list2.csv")
node3 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list3.csv")
others = pd.read_csv(r"D:\commo\code\4_kumu_struct\others.csv")

In [99]:
node1.rename(columns={
    'Label':'label',
    'Type':'role',
    'Description':'description',
    'Tier':'tier',
    'Commodity Type':'commodity_type',
    'Commodity Focus':'commodity_focus',
    'Legal Name / ACRA UEN':'acra_uen',
    'HQ Country':'hq_country',
    'Estimated Revenue':'est_revenue',
    'Headcount (Global)':'global_headcount',
    'Trade Volume (Est.)':'est_trade_volume',
    'Ownership Structure':"ownership_struct",
    'Exchange':'exchange',
    'APAC offices':'sg_address'
}, inplace=True)
node2.rename(columns={
    'Label':'label',
    'Type':'role',
    'Description':'description',
    'Scope':'scope',
    'Office Location(s) in APAC':'sg_address',
    'Practice Areas':'practice_areas',
    'Known Client Base':'known_clients'
}, inplace=True)
node3.rename(columns={
    'Label':'label',
    'Type':'role',
    'Description':'description',
    'Financier Type':'financieir_type',
    'Commodity Type':'commodity_type',
    'Geographic Reach':'geographic_reach',
    'Known Clients':'known_clients'
}, inplace=True)
others.rename(columns={
    'Label':'label',
    'Type':'role',
}, inplace=True)

In [100]:
all_documented_labels = set(node1['label']) | set(node2['label']) | set(node3['label']) | set(others['label'])
edge_labels = set(edge_list['From']) | set(edge_list['To'])

undocumented = edge_labels - all_documented_labels
print(len(undocumented), "undocumented entities out of", len(edge_labels), "total in edge_list")
print(sorted(undocumented))

0 undocumented entities out of 163 total in edge_list
[]


In [101]:
# create a mapping of unique node1 'label' indices from range [0, num_node1]
# create a mapping of unique node2 'label' indices from range [0, num_node2]
# create a mapping of unique node3 'label' indices from range [0, num_node3]
# create a mapping of unique others 'label' indices from range [0, num_others]

In [102]:
# create a mapping of unique node1 'label' indices from range [0, num_node1]
unique_node1_label = node1['label'].unique()
unique_node1_label = pd.DataFrame(data={
    'node1_label': unique_node1_label,
    'node1_mappedID': pd.RangeIndex(len(unique_node1_label)),
})
print("Mapping of node1 labels to consecutive values:")
print("==========================================")
print(unique_node1_label.head())
print()

# create a mapping of unique node2 'label' indices from range [0, num_node2]
unique_node2_label = node2['label'].unique()
unique_node2_label = pd.DataFrame(data={
    'node2_label': unique_node2_label,
    'node2_mappedID':pd.RangeIndex(len(unique_node2_label))
})
print('Mapping of node2 labels to consecutive values:')
print("==========================================")
print(unique_node2_label.head())
print()

# create a mapping of unique node3 'label' indices from range [0, num_node3]
unique_node3_label = node3['label'].unique()
unique_node3_label = pd.DataFrame(data={
    'node3_label': unique_node3_label,
    'node3_mappedID':pd.RangeIndex(len(unique_node3_label))
})
print('Mapping of node3 labels to consecutive values:')
print("==========================================")
print(unique_node3_label.head())
print()

# create a mapping of unique others 'label' indices from range [0, num_others]
unique_others_label = others['label'].unique()
unique_others_label = pd.DataFrame(data={
    'others_label': unique_others_label,
    'others_mappedID':pd.RangeIndex(len(unique_others_label))
})
print('Mapping of others labels to consecutive values:')
print("==========================================")
print(unique_others_label.head())
print()

Mapping of node1 labels to consecutive values:
                  node1_label  node1_mappedID
0             trafigura group               0
1                  vitol asia               1
2           mercuria holdings               2
3            gunvor singapore               3
4  louis dreyfus company asia               4

Mapping of node2 labels to consecutive values:
                 node2_label  node2_mappedID
0                 reed smith               0
1  holman fenwick willan hfw               1
2                 clyde & co               2
3         stephenson harwood               3
4   watson farley & williams               4

Mapping of node3 labels to consecutive values:
          node3_label  node3_mappedID
0                 ing               0
1    societe generale               1
2         bnp paribas               2
3            rabobank               3
4  standard chartered               4

Mapping of others labels to consecutive values:
                             other

In [103]:
# node1 input features: role, tier, commodity_type, hq_country, ownership_struct, exchange
node1['node1_mappedID'] = node1['label'].map(unique_node1_label.set_index('node1_label')['node1_mappedID'])
node1.set_index('node1_mappedID', inplace=True)

# split role, tier, commodity_type, hq_country, ownership_struct, exchange into indicator variables
node1["role"] = node1["role"].str.replace(r"\s*,\s*", ",", regex=True)  # normalize spacing, if any
node1_role = node1["role"].str.get_dummies(",")

node1["tier"] = pd.to_numeric(node1["tier"], errors="coerce")
node1["tier"] = node1["tier"].fillna(node1["tier"].median()) # fill nan with 2
node1["tier"] = (node1["tier"] - node1["tier"].min()) / (node1["tier"].max() - node1["tier"].min())

node1["commodity_type"] = node1["commodity_type"].str.replace(r"\s*\|\s*", "|", regex=True)
node1_commodity_type = node1["commodity_type"].str.get_dummies("|")

node1["hq_country"] = node1["hq_country"].str.replace(r"\s*\|\s*", "|", regex=True)
node1_hq_country = node1["hq_country"].str.get_dummies("|")

node1["ownership_struct"] = node1["ownership_struct"].str.replace(r"\s*\|\s*", "|", regex=True)
node1_ownership_struct = node1["ownership_struct"].str.get_dummies("|")

node1["exchange"] = node1["exchange"].str.replace(r"\s*\|\s*", "|", regex=True)
node1_exchange = node1["exchange"].str.get_dummies("|")

# concat columns on mappedID
node1_one_hot = pd.concat([node1_role, node1['tier'], node1_commodity_type, node1_hq_country, node1_ownership_struct, node1_exchange],axis=1)
assert node1_one_hot.shape[0] == 135
# (135,49)
# use all one-hot encoded columns as features
node1_feat = torch.from_numpy(node1_one_hot.values).to(torch.float)
assert not torch.isnan(node1_feat).any()

In [104]:
# node2 input features: role, scope, practice_areas
node2['node2_mappedID'] = node2['label'].map(unique_node2_label.set_index('node2_label')['node2_mappedID'])
node2.set_index('node2_mappedID', inplace=True)

# split role, scope, practice_areas into indicator variables
node2_role = pd.get_dummies(node2["role"]).astype(int)

node2_scope = node2["scope"].str.get_dummies("|")

node2["practice_areas"] = node2["practice_areas"].str.replace(r"\s*\|\s*", "|", regex=True)
node2_practice_areas = node2["practice_areas"].str.get_dummies("|")

# concat columns on mappedID
node2_one_hot = pd.concat([node2_role, node2_scope, node2_practice_areas], axis=1)
assert node2_one_hot.shape[0] == 39
# (39,34)
# use all one-hot encoded columns as features
node2_feat = torch.from_numpy(node2_one_hot.values).to(torch.float)

In [105]:
# node3 input features: role, financier_type, commodity_type (row: 73)
node3['node3_mappedID'] = node3['label'].map(unique_node3_label.set_index('node3_label')['node3_mappedID'])
node3.set_index('node3_mappedID', inplace=True)
# run once then comment out
node3 = node3.drop(columns=['geographic_reach', 'known_clients'])
# split role, financier_type, commodity_type into indicator variables
node3_role = pd.get_dummies(node3["role"]).astype(int)

node3_financier_type = node3["financier_type"].str.get_dummies("|")

node3["commodity_type"] = node3["commodity_type"].str.replace(r"\s*\|\s*", "|", regex=True)
node3_commodity_type = node3["commodity_type"].str.get_dummies("|")

node3["description"] = node3["description"].str.strip()
node3_description = node3["description"].str.get_dummies("|")

# concat columns on mappedID
node3_one_hot = pd.concat([node3_role, node3_financier_type, node3_commodity_type, node3_description], axis=1)
assert node3_one_hot.shape[0] == 73
# use all one-hot encoded columns as features
node3_feat = torch.from_numpy(node3_one_hot.values).to(torch.float)

In [106]:
# others input features: role (48)
others['others_mappedID'] = others['label'].map(unique_others_label.set_index('others_label')['others_mappedID'])
others.set_index("others_mappedID", inplace=True)

# split role indicator variables
others_role = pd.get_dummies(others["role"]).astype(int)

others_one_hot = others_role
assert others_one_hot.shape[0] == 48
# (48,1)
# use all one-hot encoded colum as feature
others_feat = torch.from_numpy(others_one_hot.values).to(torch.float)
assert others_feat.size() == (48, 1)

In [107]:
# combined lookup
combined_lookup = pd.concat(
    [
        unique_node1_label.set_index("node1_label")["node1_mappedID"].apply(lambda x: ("trader", x)),
        unique_node2_label.set_index("node2_label")["node2_mappedID"].apply(lambda x: ("lawyer", x)),
        unique_node3_label.set_index("node3_label")["node3_mappedID"].apply(lambda x: ("lender", x)),
        unique_others_label.set_index("others_label")["others_mappedID"].apply(lambda x: ("others", x)),
    ]
)
# check duplicates in combine_lookup
#combined_lookup[combined_lookup.index.duplicated(keep=False)]

edge_list['from_node_type'], edge_list['from_node_id'] = zip(*edge_list['From'].map(combined_lookup))
edge_list['to_node_type'], edge_list['to_node_id'] = zip(*edge_list['To'].map(combined_lookup))

In [108]:
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

data = HeteroData()
# Save unique node indices:
data['trader'].node_id = torch.from_numpy(unique_node1_label['node1_mappedID'].values)
data["lawyer"].node_id = torch.from_numpy(unique_node2_label["node2_mappedID"].values)
data["lender"].node_id = torch.from_numpy(unique_node3_label["node3_mappedID"].values)
data["others"].node_id = torch.from_numpy(unique_others_label["others_mappedID"].values)

# add node features and edge indices
data['trader'].x = node1_feat
data['lawyer'].x = node2_feat
data['lender'].x = node3_feat
data['others'].x = others_feat

for (from_node_type, edge_type, to_node_type), group in edge_list.groupby(["from_node_type", 'Type', "to_node_type"]):
    # print(group)
    # print(group['from_node_id])
    # print(group['from_node_id'].values)
    # print(torch.from_numpy(group["from_node_id"].values))
    src = torch.from_numpy(group["from_node_id"].values)
    target = torch.from_numpy(group["to_node_id"].values)
    data[from_node_type, edge_type, to_node_type].edge_index = torch.stack([src, target], dim=0)

# We won't add reverse edges because it is directed graph
data
#print(data.edge_types)

HeteroData(
  trader={
    node_id=[135],
    x=[135, 49],
  },
  lawyer={
    node_id=[39],
    x=[39, 34],
  },
  lender={
    node_id=[73],
    x=[73, 24],
  },
  others={
    node_id=[48],
    x=[48, 1],
  },
  (lawyer, counsels_for, lender)={ edge_index=[2, 15] },
  (lawyer, counsels_for, others)={ edge_index=[2, 34] },
  (lawyer, counsels_for, trader)={ edge_index=[2, 29] },
  (lender, lends_to, others)={ edge_index=[2, 9] },
  (lender, lends_to, trader)={ edge_index=[2, 48] },
  (lender, owns, lender)={ edge_index=[2, 3] },
  (others, owns, trader)={ edge_index=[2, 1] },
  (trader, lends_to, others)={ edge_index=[2, 7] },
  (trader, lends_to, trader)={ edge_index=[2, 2] },
  (trader, owns, others)={ edge_index=[2, 1] },
  (trader, owns, trader)={ edge_index=[2, 21] }
)

In [109]:
# split edges into training, validation, test splits
# use transforms.RandomLinkSplit from PyG to randomly divide the edges into training, validation and test edges.
# transforms.RandomLinkSplit's disjoint_train_ratio param to separates edges in the training split into 2: training message edges (edge_index) & training supervision edges (edge_label_index)
# 170 edges from len(edge_list)
# training (80%): 70% training message passing edges + 30% training supervision edges, validation edges (10%), and testing edges (10%).
# Generate fixed negative edges for evaluation with a ratio of 2:1.
# Negative edges during training will be generated on-the-fly.
torch.manual_seed(42)
transform = T.RandomLinkSplit(
    num_val=0.1,  # 10% val
    num_test=0.1,  # 10% test
    disjoint_train_ratio=0.3,  # 30% out of 80% training edges for training supervision
    neg_sampling_ratio=2.0,  # 2:1
    add_negative_train_samples=False,
    edge_types=data.edge_types,
    rev_edge_types=None,
)
train_data, val_data, test_data = transform(data)
# 6 out of 11 edge types have 0 validation edges
# limitation:
# 6 of 11 edge-type combinations:(lender, lends_to, others), (lender, owns, lender), (others, owns, trader), (trader, lends_to, others), (trader, lends_to, trader), (trader, owns, others) > have too few observed edges (1–9) for a standard 80/10/10 split to allocate any validation or test edges;
# these relations were retained in the graph as message-passing signal but could not be formally evaluated.
# reflects data scarcity of the confidential network itself, not a limitation of the splitting method.
# Future work could explore leave-one-out cross-validation for these relations specifically, though with as few as 1–3 total edges, some relations (others, owns, trader; trader, owns, others) cannot be meaningfully validated by any method and would require additional data collection to assess.

# if val_data[rel].edge_label_index.shape[1] == 0, skip evaluation for this relation

In [110]:
# rgcn model
# count total num of nodes across all node types to get global IDs
node_types = ['trader','lawyer','lender','others']
node_offsets = {}
offset = 0
for ntype in node_types:
    n = data[ntype].num_nodes # no. of nodes per node type
    node_offsets[ntype] = offset # node_offsets[node_type] = 0
    offset += n
total_nodes = offset # 0 + 135 + 39 + 73 + 48 = 295
node_offsets['trader'] # trader node id starts at 0
node_offsets["lawyer"]  # lawyer node id starts at 135
node_offsets # dict with each node type and its starting id

# assign each relation an integer ID
relation_names  = sorted(set(rel for (_,rel,_) in data.edge_types)) # sort alphabetically
# print(edge_names) # unique edge types
# create dictionary mapping each unique edge type to an int id
unique_edge_to_id = {rel: id for id, rel in enumerate(relation_names)}
unique_edge_to_id  # {'counsels_for': 0, 'lends_to': 1, 'owns': 2}
num_edges = len(relation_names)

# build the global edge_index and edge_type from a given split
def build_global_edges(split_data, node_offsets, unique_edge_to_id):
    src_list, dst_list, edge_type_list = [],[],[]
    for src_node_type, rel, dst_node_type in split_data.edge_types:
        # training message edges only
        edge_index = split_data[src_node_type, rel, dst_node_type].edge_index
        if edge_index.shape[1] == 0:
            continue
        # print(from_node_type) # lawyer
        # print(node_offsets[from_node_type]) # node_offsets['lawyer'] = 135
        # print(edge_index[0])
        src = edge_index[0] + node_offsets[src_node_type]
        dst = edge_index[1] + node_offsets[dst_node_type]
        src_list.append(src)
        dst_list.append(dst)
        # print(edge_type, unique_edge_to_id[edge_type])
        # print(torch.full((edge_index.shape[1],), unique_edge_to_id[edge_type], dtype=torch.long)) # size of edge_index.shape[1] filled with edge_type id
        edge_type_list.append(torch.full(size=(edge_index.shape[1],), fill_value=unique_edge_to_id[rel], dtype=torch.long))
    # print(torch.cat(src_list).shape)
    edge_index = torch.stack([torch.cat(src_list), torch.cat(dst_list)], dim=0)
    edge_type = torch.cat(edge_type_list)
    return edge_index, edge_type

# rgcn model aggregates over training message edges only
train_edge_index, train_edge_type = build_global_edges(train_data, node_offsets, unique_edge_to_id)
train_edge_index.shape  # 2,107
train_edge_type.shape  # 107

# feature projection + concat into global order
# each node type's feature .x gets its own linear layer (diff input dims per type)
# then all 4 outputs are concatenated in the same order as node_offsets so that node_offsets[node_type] in tensor correspond to global lawyer node id in edge_index
# print(data['trader'].node_id)  # all the ids of trader node 0-134
# print(data['trader'].x.shape) # all the node features size 135,49
# print(torch.nn.Linear(in_features=data['trader'].x.shape[1], out_features=64)) # in_features = no. feature columns, out_features = hidden_dim > Linear(in_features=49, out_features=64, bias=True)
hidden_dim = 64
proj = torch.nn.ModuleDict({ntype: torch.nn.Linear(in_features=data[ntype].x.shape[1], out_features=hidden_dim) for ntype in node_types})
def project_features(data, proj, node_types):
    return torch.cat([proj[ntype](data[ntype].x) for ntype in node_types], dim=0)

# rgcn encoder, no inverse relations,
# RGCNConv internally implements message + self-loop + normalization
from torch_geometric.nn import RGCNConv
import torch.nn.functional as F

class RGCNEncoder(torch.nn.Module):
    def __init__(self, hidden_dim, num_edges):
        super().__init__()
        # default starting point: 2 layers to get 2-hop receptive field and prevent over-smoothing node embeddings
        self.conv1 = RGCNConv(hidden_dim, hidden_dim, num_edges)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_edges)

    def forward(self, x, edge_index, edge_type):
        x = F.relu(self.conv1(x, edge_index, edge_type)) 
        x = self.conv2(x, edge_index, edge_type)
        return x

# relation-specific score function for decoder, each relation has L matrices (no. of layer) W_r^(1),..W_r^(L). Each relation's matrix has size d^(l+1)xd^(l) where d^(l) is hidden dim in layer l
class EdgeDecoder(torch.nn.Module):
    def __init__(self, hidden_dim, relation_names):
        super().__init__()
        self.edge_weights = torch.nn.ParameterDict({rel: torch.nn.Parameter(torch.randn(hidden_dim, hidden_dim)*0.1) for rel in relation_names})

    def forward(self, h_src, h_dst, rel_name):
        W = self.edge_weights[rel_name]
        return (h_src @ W * h_dst).sum(dim=-1)

# r-gcn model with encoder and decoder
class Model(torch.nn.Module):
    def __init__(self, feat_dims, hidden_dim, node_types, relation_names, num_edges):
        super().__init__()
        self.node_types = node_types
        self.proj = torch.nn.ModuleDict({ntype: torch.nn.Linear(feat_dims[ntype], hidden_dim) for ntype in node_types})
        self.encoder = RGCNEncoder(hidden_dim, num_edges)
        self.decoder = EdgeDecoder(hidden_dim, relation_names)

    def encode(self, data, node_offsets, edge_index, edge_type):
        x = torch.cat([self.proj[ntype](data[ntype].x) for ntype in self.node_types], dim=0)
        h = self.encoder(x, edge_index, edge_type)
        # slice back into per-type dict for easy indexing later
        h_dict = {}
        for ntype in self.node_types:
            start = node_offsets[ntype]
            end = start + data[ntype].num_nodes
            h_dict[ntype] = h[start:end]
        return h_dict

    def decode(self, h_dict, edge_label_index, src_node_type, dst_node_type, rel_name):
        h_src = h_dict[src_node_type][edge_label_index[0]]
        h_dst = h_dict[dst_node_type][edge_label_index[1]]
        return self.decoder(h_src, h_dst, rel_name)

# instantiate
feature_dimensions = {ntype: data[ntype].x.shape[1] for ntype in node_types}
# print(feature_dimensions) # no of feature columns of each node type
hidden_dim = 64
model = Model(feature_dimensions, hidden_dim, node_types, relation_names, num_edges)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [111]:
# training loop
def sample_negatives(pos_edge_index, num_dst, num_neg=2):
    num_pos = pos_edge_index.shape[1]
    neg_src = pos_edge_index[0].repeat_interleave(num_neg)
    neg_dst = torch.randint(0, num_dst, (num_pos * num_neg,))
    return torch.stack([neg_src, neg_dst], dim=0)

def train_epoch():
    model.train()
    optimizer.zero_grad()
    h_dict = model.encode(train_data, node_offsets, train_edge_index, train_edge_type)
    total_loss = 0.0
    for (src_node_type, rel, dst_node_type) in train_data.edge_types:
        pos_index = train_data[src_node_type, rel, dst_node_type].edge_label_index
        if pos_index.shape[1] == 0:
            continue
        pos_scores = model.decode(h_dict, pos_index, src_node_type, dst_node_type, rel)
        pos_labels = torch.ones(pos_index.shape[1])

        num_dst = h_dict[dst_node_type].shape[0]
        neg_index = sample_negatives(pos_index, num_dst, num_neg=2)
        neg_scores = model.decode(h_dict, neg_index, src_node_type, dst_node_type, rel)
        neg_labels = torch.zeros(neg_index.shape[1])

        scores = torch.cat([pos_scores, neg_scores])
        labels = torch.cat([pos_labels, neg_labels])
        total_loss = total_loss + F.binary_cross_entropy_with_logits(scores, labels)

    total_loss.backward()
    optimizer.step()
    return float(total_loss)

In [112]:
def build_eval_edges(train_data, node_offsets, unique_edge_to_id):
    src_list, dst_list, edge_type_list = [], [], []
    for src_type, rel, dst_type in train_data.edge_types:
        message_index = train_data[src_type, rel, dst_type].edge_index

        # pull only the POSITIVE supervision edges (label == 1), never the negatives
        sup_index_full = train_data[src_type, rel, dst_type].edge_label_index
        sup_label = train_data[src_type, rel, dst_type].edge_label
        pos_mask = sup_label == 1
        sup_index = sup_index_full[:, pos_mask]

        # concatenate message + positive supervision edges for this relation
        combined_index = torch.cat([message_index, sup_index], dim=1)
        if combined_index.shape[1] == 0:
            continue

        src = combined_index[0] + node_offsets[src_type]
        dst = combined_index[1] + node_offsets[dst_type]
        src_list.append(src)
        dst_list.append(dst)
        edge_type_list.append(torch.full((combined_index.shape[1],),
                                          unique_edge_to_id[rel], dtype=torch.long))

    edge_index = torch.stack([torch.cat(src_list), torch.cat(dst_list)], dim=0)
    edge_type = torch.cat(edge_type_list)
    return edge_index, edge_type

val_edge_index, val_edge_type = build_eval_edges(train_data, node_offsets, unique_edge_to_id)

In [113]:
# evaluation (Hits@k, Reciprocal Rank), both higher better and auc
@torch.no_grad()
def evaluate_with_metrics(split_data, edge_index, edge_type, node_offsets, k_values=(1, 3, 10),known_positives_per_head=None):
    """
    known_positives_per_head: optional dict[(src_type, rel, dst_type)][head_id] -> set(tail_ids)
                               already known true, to exclude from the ranking candidate pool
                               (Image 10's rule). Pass None to skip this filtering.
    """
    model.eval()
    h_dict = model.encode(split_data, node_offsets, edge_index, edge_type)

    results = {}
    for (src_type, rel, dst_type) in split_data.edge_types:
        edge_label_index = split_data[src_type, rel, dst_type].edge_label_index
        edge_label = split_data[src_type, rel, dst_type].edge_label
        if edge_label_index.shape[1] == 0:
            continue

        pos_mask = edge_label == 1
        heads = edge_label_index[0][pos_mask]
        tails = edge_label_index[1][pos_mask]
        num_dst = h_dict[dst_type].shape[0]

        per_triple = []
        for h_id, t_id in zip(heads.tolist(), tails.tolist()):
            candidate_ids = list(range(num_dst))

            if known_positives_per_head is not None:
                known = known_positives_per_head.get((src_type, rel, dst_type), {}).get(h_id, set())
                candidate_ids = [c for c in candidate_ids if c == t_id or c not in known]

            h_src = h_dict[src_type][h_id].unsqueeze(0).expand(len(candidate_ids), -1)
            h_dst = h_dict[dst_type][torch.tensor(candidate_ids)]
            scores = model.decoder(h_src, h_dst, rel)

            true_pos_in_list = candidate_ids.index(t_id)
            true_score = scores[true_pos_in_list]
            rank = (scores >= true_score).sum().item()

            per_triple.append({
                'head': h_id,
                'tail': t_id,
                'rank': rank,
                'reciprocal_rank': 1.0 / rank,
                **{f'hits@{k}': int(rank <= k) for k in k_values}
            })
        results[(src_type, rel, dst_type)] = per_triple
    return results

from sklearn.metrics import roc_auc_score

@torch.no_grad()
def evaluate_auc(split_data, edge_index, edge_type, node_offsets):
    model.eval()
    h_dict = model.encode(split_data, node_offsets, edge_index, edge_type)

    results = {}
    for (src_type, rel, dst_type) in split_data.edge_types:
        edge_label_index = split_data[src_type, rel, dst_type].edge_label_index
        edge_label = split_data[src_type, rel, dst_type].edge_label
        if edge_label_index.shape[1] == 0:
            continue
        # need both classes present to compute AUC
        if edge_label.unique().numel() < 2:
            print(f"{(src_type, rel, dst_type)}: skipped, only one class present")
            continue

        scores = model.decode(h_dict, edge_label_index, src_type, dst_type, rel)
        probs = torch.sigmoid(scores)
        auc = roc_auc_score(edge_label.numpy(), probs.numpy())
        results[(src_type, rel, dst_type)] = auc

    return results

In [114]:
detail_rows = []
auc_rows = []
for run in range(5):
    print(f"\n=== RUN {run+1} ===")
    model = Model(feature_dimensions, hidden_dim, node_types, relation_names, num_edges)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(1, 201):
        loss = train_epoch()
        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    results = evaluate_with_metrics(val_data, val_edge_index, val_edge_type, node_offsets, k_values=(1, 3, 10))
    for rel_key, per_triple in results.items():
        print(f"\n{rel_key}  (n={len(per_triple)})")
        for t in per_triple:
            hits_str = ", ".join(f"Hits@{k}={t[f'hits@{k}']}" for k in (1, 3, 10))
            print(f"  head={t['head']}, tail={t['tail']}, rank={t['rank']}, "
                f"RR={t['reciprocal_rank']:.3f}, {hits_str}")
            detail_rows.append({
                'run': run + 1,
                'relation': rel_key,
                'head': t['head'],
                'tail': t['tail'],
                'rank': t['rank'],
                'reciprocal_rank': t['reciprocal_rank'],
                'hits@1': t['hits@1'],
                'hits@3': t['hits@3'],
                'hits@10': t['hits@10'],
            })

    auc_results = evaluate_auc(val_data, val_edge_index, val_edge_type, node_offsets)
    for rel_key, auc in auc_results.items():
        print(f"{rel_key}: AUC = {auc:.3f}")
        auc_rows.append({
            'run': run + 1,
            'relation': rel_key,
            'auc': auc,
        })
detail_df = pd.DataFrame(detail_rows)
auc_df = pd.DataFrame(auc_rows)


=== RUN 1 ===
Epoch 10, Loss: 4.8575
Epoch 20, Loss: 3.5440
Epoch 30, Loss: 3.0270
Epoch 40, Loss: 2.6091
Epoch 50, Loss: 2.7100
Epoch 60, Loss: 2.1188
Epoch 70, Loss: 2.4632
Epoch 80, Loss: 1.9284
Epoch 90, Loss: 1.8781
Epoch 100, Loss: 1.5065
Epoch 110, Loss: 2.0739
Epoch 120, Loss: 2.1993
Epoch 130, Loss: 1.8023
Epoch 140, Loss: 2.1993
Epoch 150, Loss: 1.8281
Epoch 160, Loss: 1.2640
Epoch 170, Loss: 1.8251
Epoch 180, Loss: 1.7655
Epoch 190, Loss: 1.5409
Epoch 200, Loss: 2.2130

('lawyer', 'counsels_for', 'lender')  (n=1)
  head=16, tail=7, rank=54, RR=0.019, Hits@1=0, Hits@3=0, Hits@10=0

('lawyer', 'counsels_for', 'others')  (n=3)
  head=24, tail=27, rank=5, RR=0.200, Hits@1=0, Hits@3=0, Hits@10=1
  head=30, tail=22, rank=5, RR=0.200, Hits@1=0, Hits@3=0, Hits@10=1
  head=34, tail=24, rank=36, RR=0.028, Hits@1=0, Hits@3=0, Hits@10=0

('lawyer', 'counsels_for', 'trader')  (n=2)
  head=9, tail=0, rank=20, RR=0.050, Hits@1=0, Hits@3=0, Hits@10=0
  head=34, tail=57, rank=107, RR=0.009,

In [115]:
# save auc_df and detail_df
auc_df.to_csv(r"D:\commo\code\link_prediction\auc_df.csv", index=False)
detail_df.to_csv(r"D:\commo\code\link_prediction\detail_df.csv", index=False)

In [116]:
auc_summary = auc_df.groupby('relation')['auc'].agg(
    n_val_triples=lambda x: 'see detail table',  # or pull from detail_df counts
    mean_auc='mean',
    min_auc='min',
    max_auc='max'
).round(3)
n_triples = detail_df.groupby('relation')['head'].count() / 5  # divide by 5 runs to get per-run count
auc_summary = auc_df.groupby('relation')['auc'].agg(['mean', 'min', 'max']).round(3)
auc_summary.insert(0, 'n_validation_triples', n_triples.astype(int))
print(auc_summary)

pivot_rank = detail_df.pivot_table(
    index=['relation', 'head', 'tail'],
    columns='run',
    values='rank'
)
pivot_rank.columns = [f'rank_run{c}' for c in pivot_rank.columns]
pivot_rank_flat = pivot_rank.reset_index()
pivot_rank
# save auc_summary and pivot_rank
auc_summary.to_csv(r"D:\commo\code\link_prediction\auc_summary.csv", index=False)
pivot_rank_flat.to_csv(r"D:\commo\code\link_prediction\pivot_rank_flat.csv", index=False)

                                n_validation_triples   mean    min    max
relation                                                                 
(lawyer, counsels_for, lender)                     1  0.900  0.500  1.000
(lawyer, counsels_for, others)                     3  0.867  0.833  0.944
(lawyer, counsels_for, trader)                     2  0.400  0.250  0.500
(lender, lends_to, trader)                         4  0.831  0.750  0.906
(trader, owns, trader)                             2  0.550  0.375  0.750


In [117]:
pivot_rank_flat

,relation,head,tail,rank_run1,rank_run2,rank_run3,rank_run4,rank_run5
0,"(lawyer, counsels_for, lender)",16,7,54.0,12.0,46.0,65.0,10.0
1,"(lawyer, counsels_for, others)",24,27,5.0,5.0,5.0,5.0,5.0
2,"(lawyer, counsels_for, others)",30,22,5.0,5.0,5.0,5.0,5.0
3,"(lawyer, counsels_for, others)",34,24,36.0,20.0,35.0,31.0,26.0
4,"(lawyer, counsels_for, trader)",9,0,20.0,98.0,46.0,59.0,98.0
5,"(lawyer, counsels_for, trader)",34,57,107.0,122.0,124.0,118.0,123.0
6,"(lender, lends_to, trader)",5,5,9.0,3.0,4.0,4.0,5.0
7,"(lender, lends_to, trader)",5,19,26.0,27.0,31.0,26.0,27.0
8,"(lender, lends_to, trader)",12,125,6.0,5.0,5.0,5.0,7.0
9,"(lender, lends_to, trader)",18,129,73.0,50.0,39.0,73.0,89.0


In [118]:
def get_node_degree(node_id, node_type, train_data):
    count = 0
    for rel_key in train_data.edge_types:
        if rel_key[0] == node_type:
            ei = train_data[rel_key].edge_index
            count += (ei[0] == node_id).sum().item()
    return count

@torch.no_grad()
def predict_missing_links(model, data, node_offsets, edge_index, edge_type,
                           src_type, rel, dst_type, top_k=5, train_data=None,
                           val_data=None, test_data=None, min_degree=0):
    model.eval()
    h_dict = model.encode(data, node_offsets, edge_index, edge_type)

    known_pairs = set()
    for split in [train_data, val_data, test_data]:
        for key in [(src_type, rel, dst_type)]:
            if key in split.edge_types:
                ei = split[key].edge_index
                known_pairs.update(zip(ei[0].tolist(), ei[1].tolist()))
                if hasattr(split[key], 'edge_label_index'):
                    lbl = split[key].edge_label
                    eli = split[key].edge_label_index
                    pos = eli[:, lbl == 1]
                    known_pairs.update(zip(pos[0].tolist(), pos[1].tolist()))

    num_src = h_dict[src_type].shape[0]
    num_dst = h_dict[dst_type].shape[0]

    # precompute degree for every src node ONCE, not per-prediction
    degrees = {}
    if min_degree > 0:
        for h_id in range(num_src):
            degrees[h_id] = get_node_degree(h_id, src_type, train_data)

    predictions = []
    for h_id in range(num_src):
        if min_degree > 0 and degrees.get(h_id, 0) < min_degree:
            continue  # skip src nodes below the degree threshold

        h_src = h_dict[src_type][h_id].unsqueeze(0).expand(num_dst, -1)
        scores = model.decoder(h_src, h_dict[dst_type], rel)
        probs = torch.sigmoid(scores)

        for t_id in range(num_dst):
            if (h_id, t_id) in known_pairs:
                continue
            predictions.append((h_id, t_id, probs[t_id].item()))

    predictions.sort(key=lambda x: x[2], reverse=True)
    return predictions[:top_k]

In [119]:
lender_names = unique_node3_label.set_index("node3_mappedID")["node3_label"]
trader_names = unique_node1_label.set_index("node1_mappedID")["node1_label"]

top_predictions = predict_missing_links(
    model, data, node_offsets, val_edge_index, val_edge_type,
    src_type='lender', rel='lends_to', dst_type='trader', top_k=10,
    train_data=train_data, val_data=val_data, test_data=test_data,
    min_degree=2   # excludes lender 42 (degree=1) and anything else with <2 real edges
)

def get_node_degree_as_target(node_id, node_type, train_data):
    count = 0
    for rel_key in train_data.edge_types:
        if rel_key[2] == node_type:  # count edges pointing INTO this node
            ei = train_data[rel_key].edge_index
            count += (ei[1] == node_id).sum().item()
    return count

pred_src_target_edge = []
for h_id, t_id, prob in top_predictions:
    lender_deg = get_node_degree(h_id, 'lender', train_data)
    trader_deg = get_node_degree_as_target(t_id, 'trader', train_data)
    print(f"{lender_names[h_id]} --lends_to--> {trader_names[t_id]} "
          f"(prob={prob:.3f}, lender_degree={lender_deg}, trader_degree={trader_deg})")
    pred_src_target_edge.append({'From':lender_names[h_id], 'To':trader_names[t_id], 'Direction':'directed','Type':'lends_to'})
pred_edges = pd.DataFrame(pred_src_target_edge)
new_edge_list_w_pred_edges = pd.concat([edge_list, pred_edges], ignore_index=True)

# save the new edge list with predicted edge for network visualization
new_edge_list_w_pred_edges.to_csv(r'D:\commo\code\link_prediction\new_edge_list_w_pred_edges.csv', index=False)

united overseas bank uob --lends_to--> olam food ingredients (prob=0.986, lender_degree=2, trader_degree=0)
ing --lends_to--> olam food ingredients (prob=0.977, lender_degree=3, trader_degree=0)
mufg --lends_to--> olam food ingredients (prob=0.977, lender_degree=6, trader_degree=0)
hsbc --lends_to--> olam food ingredients (prob=0.977, lender_degree=3, trader_degree=0)
natixis --lends_to--> olam food ingredients (prob=0.977, lender_degree=2, trader_degree=0)
dbs bank --lends_to--> olam food ingredients (prob=0.962, lender_degree=3, trader_degree=0)
ing --lends_to--> olam agri holdings (prob=0.959, lender_degree=3, trader_degree=0)
mufg --lends_to--> olam agri holdings (prob=0.959, lender_degree=6, trader_degree=0)
natixis --lends_to--> olam agri holdings (prob=0.959, lender_degree=2, trader_degree=0)
ing --lends_to--> mercuria energy group (prob=0.943, lender_degree=3, trader_degree=0)


In [120]:
# Training converges by approximately 200 epochs at lr=0.001; 
# additional training up to 500 epochs produced no meaningful change in validation rankings, confirming the model reached a stable solution rather than continuing to improve or degrade. 
# Given the small validation set sizes (n=1 to n=4 per relation), AUC and reciprocal rank should be interpreted with corresponding caution — 
# the lends_to→trader relation, with the most validation support (n=4), shows the most reliable evidence of learned structure (AUC=0.875). 
# Relations with only 1–2 validation triples (counsels_for→lender, counsels_for→trader, owns→trader) yield AUC values that are consistent across converged runs, but the underlying sample sizes are too small to treat these as statistically robust estimates of true performance.

In [121]:
# lends_to→trader achieved AUC = 0.70 (range 0.6–0.84) across 5 independent training runs on a fixed validation split, the strongest and most reproducible result among the evaluated relations.
"""

**Finding: Some high-confidence predictions point to companies that look "unconnected" only because of how the data is structured, not because the relationship doesn't exist.**

Several of the model's top predictions involve well-known trading companies like Olam and Mercuria. At first glance, these companies appeared to have zero recorded lending relationships in the training data, which seemed odd since we know both are large, real, well-connected companies.

On closer inspection, the real explanation is simpler: **large companies are split across multiple nodes in the dataset** — one node for the parent company, separate nodes for subsidiaries, and sometimes separate nodes for slightly different name spellings of the same business. For example, "Mercuria Energy Group," "Mercuria Energy Trading," and "Mercuria Holdings" are three different nodes in the graph, even though in reality they're all part of the same company.

So when the model predicts that a bank might lend to "Mercuria Energy Group," it's not necessarily wrong or inventing something from nothing — it may be recognizing that this specific corporate entity looks similar to other well-connected entities, even though *that exact node* doesn't yet have a recorded lending edge. Meanwhile, a *different* node representing basically the same company (like "Mercuria Energy Trading") does have a real recorded edge.

**In short:** the model's predictions are reasonable and consistent with real-world knowledge, but the underlying data treats one real company as several separate nodes. This means some "zero-connection" predictions aren't really about unconnected companies — they're about a company whose data got split up. This is a data structure issue worth flagging, not a sign the model is guessing randomly.

**Why this matters going forward:** if we combined parent companies and subsidiaries into single nodes (or explicitly linked them with a "subsidiary of" relationship), the model would likely have access to more complete information about these large companies, and its predictions might become even more accurate. This is a good candidate for future improvement.
"""

'\n\n**Finding: Some high-confidence predictions point to companies that look "unconnected" only because of how the data is structured, not because the relationship doesn\'t exist.**\n\nSeveral of the model\'s top predictions involve well-known trading companies like Olam and Mercuria. At first glance, these companies appeared to have zero recorded lending relationships in the training data, which seemed odd since we know both are large, real, well-connected companies.\n\nOn closer inspection, the real explanation is simpler: **large companies are split across multiple nodes in the dataset** — one node for the parent company, separate nodes for subsidiaries, and sometimes separate nodes for slightly different name spellings of the same business. For example, "Mercuria Energy Group," "Mercuria Energy Trading," and "Mercuria Holdings" are three different nodes in the graph, even though in reality they\'re all part of the same company.\n\nSo when the model predicts that a bank might lend 

In [122]:
# create a directed graph with the predicted edges
import networkx as nx
import pandas as pd

new_edge_list_w_pred_edges = pd.read_csv(r'D:\commo\code\link_prediction\new_edge_list_w_pred_edges.csv')
DG = nx.DiGraph()

# add all nodes
records1 = node1.set_index("label").to_dict("index")
records2 = node2.set_index("label").to_dict("index")
records3 = node3.set_index("label").to_dict("index")
records4 = others.set_index("label").to_dict("index")
DG.add_nodes_from(records1.items())
DG.add_nodes_from(records2.items())
DG.add_nodes_from(records3.items())
DG.add_nodes_from(records4.items())

# add edges
edges = [
    (row.From, row.To, {"type": row.Type})
    for row in new_edge_list_w_pred_edges[["From", "To", "Type"]].itertuples()
]
DG.add_edges_from(edges)  # full edge list

# collective metric of the directed graph
# number of nodes and edges
# node count
total_num_nodes = nx.number_of_nodes(DG)
# edge count
total_num_edges = nx.number_of_edges(DG)
print('Original number of nodes and edges: ',total_num_nodes, total_num_edges)
# density of directed graph
print()
density_dg = nx.density(DG) # visualize
# checks if a directed graph is weakly connected nx.is_weakly_connected
is_wcc = nx.is_weakly_connected(DG) # false 
# no of wcc nx.number_weakly_connected_components
num_wcc = nx.number_weakly_connected_components(DG)

# save the collective metric table
collective_metric = pd.DataFrame({'Collective Metrics': ['total_num_nodes','total_num_edges','density','is_weakly_connected','num_wcc'],
                                  'Value':[total_num_nodes,total_num_edges,round(density_dg,4),is_wcc,num_wcc]})

collective_metric

Original number of nodes and edges:  295 180



,Collective Metrics,Value
0,total_num_nodes,295
1,total_num_edges,180
2,density,0.0021
3,is_weakly_connected,False
4,num_wcc,150
